In [3]:
!pip install langchain langchain-community faiss-cpu sentence-transformers transformers accelerate


In [2]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import pipeline
from sentence_transformers import SentenceTransformer
import numpy as np
import re
from textwrap import shorten

In [4]:
loader = TextLoader("story.txt")
docs = loader.load()

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# Create embeddings and FAISS vectorstore
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

# Build retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

/tmp/ipython-input-2832785703.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# 4. Load language model (FLAN-T5-BASE)
# ============================================================
flan_pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=0,                  # set to -1 if no GPU
    max_new_tokens=128,
    temperature=0.0,
    do_sample=False
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [6]:
# 5. Helpers: extractive fallback + debug_and_answer
# ============================================================
embedder = SentenceTransformer("all-MiniLM-L6-v2")
_sentence_split_re = re.compile(r'(?<=[.!?])\s+')

def extractive_fallback(context_text, question, top_n_sentences=3):
    q_tokens = set([t.lower() for t in re.findall(r"\w+", question) if len(t) > 2])
    sentences = [s.strip() for s in _sentence_split_re.split(context_text) if s.strip()]
    scores = []
    for s in sentences:
        s_tokens = set([t.lower() for t in re.findall(r"\w+", s)])
        overlap = len(q_tokens & s_tokens)
        scores.append((overlap, s))
    scores.sort(reverse=True, key=lambda x: x[0])
    selected = [s for sc, s in scores if sc > 0][:top_n_sentences]
    if not selected and sentences:
        return sentences[0]
    return " ".join(selected) if selected else ""

def debug_and_answer(query,
                     vectorstore=None,
                     retriever=None,
                     llm_pipeline=None,
                     k=3,
                     max_new_tokens=64,
                     sem_sim_threshold=0.55):
    assert llm_pipeline is not None, "llm_pipeline is required."

    # 1) Retrieve top-k
    docs_with_scores = None
    if vectorstore is not None:
        docs_with_scores = vectorstore.similarity_search_with_score(query, k=k)
    elif retriever is not None:
        docs = retriever.get_relevant_documents(query)[:k]
        docs_with_scores = [(d, None) for d in docs]
    else:
        raise ValueError("Provide either vectorstore or retriever")

    # Build context
    context_pieces = []
    for i, (doc, score) in enumerate(docs_with_scores):
        snippet = doc.page_content.strip()
        header = f"[DOC {i+1} | score={score:.4f}]\n" if score is not None else f"[DOC {i+1}]\n"
        context_pieces.append(header + snippet)
    context_text = "\n\n".join(context_pieces)

    # 2) Prompt
    prompt = f"""Context:
{context_text}

Question: {query}

Answer (use only the context; if not present answer exactly: "I don't know"):"""

    print("\n=== PROMPT ===")
    print(shorten(prompt, width=1500, placeholder="...[truncated]"))
    print("=== END PROMPT ===\n")

    # 3) Call LLM
    out = llm_pipeline(prompt, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0, top_p=1.0)
    raw_answer = out[0].get("generated_text") or out[0].get("text") or str(out[0])
    raw_answer = raw_answer.strip()
    print("Raw model output:", raw_answer[:400], ("\n... (truncated)" if len(raw_answer) > 400 else ""))

    # 4) Semantic similarity check
    doc_texts = [doc.page_content for doc, _ in docs_with_scores]
    if doc_texts:
        emb_all = embedder.encode([raw_answer] + doc_texts, convert_to_numpy=True)
        a_emb = emb_all[0]
        doc_embs = emb_all[1:]
        sims = (doc_embs @ a_emb) / (np.linalg.norm(doc_embs, axis=1) * np.linalg.norm(a_emb) + 1e-12)
        max_sim = float(np.max(sims))
        max_idx = int(np.argmax(sims))
        supporting_doc = doc_texts[max_idx]
    else:
        max_sim = 0.0
        supporting_doc = ""

    print(f"Max semantic sim(answer, top_docs) = {max_sim:.3f}")

    # 5) Fallback if unsupported
    if max_sim < sem_sim_threshold:
        extractive = extractive_fallback(" ".join(doc_texts), query)
        if extractive:
            final_answer = extractive
            note = "fallback_extractive"
        else:
            final_answer = "I don't know."
            note = "no_support"
    else:
        final_answer = raw_answer
        note = "supported"

    return {
        "final_answer": final_answer,
        "note": note,
        "support_score": max_sim,
        "supporting_doc": supporting_doc
    }


In [7]:
# ============================================================
# 6. Test queries
# ============================================================
queries = [
    "What was the boy’s name in the story?",
    "Who was Twink?",
    "What did Arjun pack in his backpack to help Twink?",
]

for q in queries:
    print("="*80)
    print("QUESTION:", q)
    result = debug_and_answer(q, vectorstore=vectorstore, llm_pipeline=flan_pipe, k=3)
    print("FINAL ANSWER:", result["final_answer"])
    print("NOTE:", result["note"], "| support_score:", result["support_score"])


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION: What was the boy’s name in the story?

=== PROMPT ===
Context: [DOC 1 | score=1.0094] The Little Star and the Brave Boy Once upon a time, in a small village, there lived a boy named Arjun. Arjun loved to look at the night sky. Every night, he waved at the stars and whispered, “Goodnight, friends!” One evening, a tiny star twinkled brighter than the others. “Hello, Arjun!” the star whispered. Arjun rubbed his eyes. “Did you just talk?” “Yes!” said the star. “My name is Twink.” Twink was sad. “I have fallen from the sky. I don’t know how to go back.” [DOC 2 | score=1.3991] They chirped, “We’ll carry Twink!” Together, the birds lifted Twink gently. Higher and higher they flew. Arjun waved from below. “Goodbye, Twink! Shine bright!” At last, Twink reached the sky. He twinkled happily. “Thank you, Arjun!” That night, Arjun looked up. Twink sparkled extra bright, just for him. Whenever Arjun felt lonely, Twink winked down. It was their secret friendship forever. And so, the boy and

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw model output: Arjun 
Max semantic sim(answer, top_docs) = 0.521
FINAL ANSWER: The Little Star and the Brave Boy 
Once upon a time, in a small village, there lived a boy named Arjun. “My name is Twink.” 
Twink was sad. And so, the boy and the star remained best friends, 
One on the Earth, 
One in the sky, Arjun smiled.
NOTE: fallback_extractive | support_score: 0.520502507686615
QUESTION: Who was Twink?

=== PROMPT ===
Context: [DOC 1 | score=0.9879] They chirped, “We’ll carry Twink!” Together, the birds lifted Twink gently. Higher and higher they flew. Arjun waved from below. “Goodbye, Twink! Shine bright!” At last, Twink reached the sky. He twinkled happily. “Thank you, Arjun!” That night, Arjun looked up. Twink sparkled extra bright, just for him. Whenever Arjun felt lonely, Twink winked down. It was their secret friendship forever. And so, the boy and the star remained best friends, One on the Earth, One in the sky, [DOC 2 | score=1.0641] The Little Star and the Brave Boy Once u

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw model output: a boy named Arjun 
Max semantic sim(answer, top_docs) = 0.612
FINAL ANSWER: a boy named Arjun
NOTE: supported | support_score: 0.612480103969574
QUESTION: What did Arjun pack in his backpack to help Twink?

=== PROMPT ===
Context: [DOC 1 | score=0.6279] Arjun smiled. “Don’t worry, Twink. I’ll help you!” The next morning, Arjun packed his backpack. He put in a ladder, a balloon, and a sandwich. “Let’s try!” he said. First, Arjun used the ladder. He climbed and climbed… but the sky was too far. Next, he tied the balloon to Twink. The balloon floated, but only a little. Twink looked worried. “What now?” Arjun thought hard. “Maybe the birds can help!” He called his bird friends—sparrows, pigeons, and parrots. [DOC 2 | score=0.8205] They chirped, “We’ll carry Twink!” Together, the birds lifted Twink gently. Higher and higher they flew. Arjun waved from below. “Goodbye, Twink! Shine bright!” At last, Twink reached the sky. He twinkled happily. “Thank you, Arjun!” That night